In [5]:
import pandas as pd
import requests
import time
from tqdm import tqdm

df = pd.read_csv("politicians_1900_1950.csv")
print(f"Politicians in all: {len(df)}")
print(f"With Wikipedia-URL: {df['wikipedia_url'].notna().sum()}")


def fetch_wikipedia_text(url: str, session=None) -> str:
    if session is None:
        session = requests.Session()
        session.headers.update({"User-Agent": "PoliticiansNetwork/1.0"})

    title = url.rstrip("/").split("/wiki/")[-1]
    params = {
        "action":          "query",
        "titles":          title,
        "prop":            "extracts",
        "explaintext":     True,
        "exsectionformat": "plain",
        "format":          "json",
    }
    r = session.get("https://en.wikipedia.org/w/api.php", params=params, timeout=20)
    r.raise_for_status()
    pages = r.json()["query"]["pages"]
    page  = next(iter(pages.values()))
    return page.get("extract", "") or ""


def build_text_corpus(df, delay=0.3):
    corpus = {}
    session = requests.Session()
    session.headers.update({"User-Agent": "PoliticiansNetwork/1.0"})

    
    for _, row in tqdm(df.head(20).iterrows(), total=20, desc="Fetching article text"):
        qid = row["wikidata"]
        url = row.get("wikipedia_url")

        if not url or pd.isna(url):
            corpus[qid] = ""
            continue

        for attempt in range(5):
            try:
                text = fetch_wikipedia_text(url, session=session)
                corpus[qid] = text
                break
            except Exception as e:
                if attempt == 4:
                    print(f"  Giving up on {url}: {e}")
                    corpus[qid] = ""
                else:
                    time.sleep(2 ** attempt)

        time.sleep(delay)

    return corpus



corpus = build_text_corpus(df)

df["article_text"] = df["wikidata"].map(corpus)


df.to_csv("politicians_1900_1950_with_text.csv", index=False)

n_with = (df["article_text"].str.len() > 0).sum()
n_without = (df["article_text"].str.len() == 0).sum()
print(f"\nText fetched:    {n_with} politicians")
print(f"No text/URL: {n_without} politicians")
print(f"\nSaved as: politicians_1900_1950_with_text.csv")

Politicians in all: 2444
With Wikipedia-URL: 2441


Fetching article text: 100%|██████████| 20/20 [00:12<00:00,  1.65it/s]


Text fetched:    19 politicians
No text/URL: 1 politicians

Saved as: politicians_1900_1950_with_text.csv
